# Advanced Time Series Forecasting Models

## Complete Forecasting Suite with Multiple Models

This notebook implements various advanced forecasting techniques:
- ARIMA/SARIMA (Statistical Models)
- Prophet (Facebook's Forecasting Tool)
- LSTM (Deep Learning)
- XGBoost (Gradient Boosting)
- Ensemble Methods
- Hybrid Models

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Time series models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pmdarima as pm
from prophet import Prophet

# Machine learning
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input, Bidirectional, Attention
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 1. Advanced Forecaster Class

In [ ]:
class AdvancedForecaster:
    """Advanced time series forecasting with multiple models."""
    
    def __init__(self, train_data: pd.Series, test_data: pd.Series = None, 
                 forecast_horizon: int = 30, confidence_level: float = 0.95):
        """Initialize the forecaster.
        
        Parameters:
        -----------
        train_data : pd.Series
            Training time series data
        test_data : pd.Series, optional
            Test time series data for evaluation
        forecast_horizon : int
            Number of periods to forecast
        confidence_level : float
            Confidence level for prediction intervals
        """
        self.train = train_data.dropna()
        self.test = test_data.dropna() if test_data is not None else None
        self.forecast_horizon = forecast_horizon
        self.confidence_level = confidence_level
        
        self.models = {}
        self.forecasts = {}
        self.prediction_intervals = {}
        self.metrics = {}
        
    def fit_all_models(self, verbose: bool = True):
        """Fit all forecasting models."""
        
        if verbose:
            print("Training forecasting models...")
        
        # Statistical Models
        if verbose:
            print("\n1. Training ARIMA model...")
        self._fit_auto_arima()
        
        if verbose:
            print("\n2. Training Prophet model...")
        self._fit_prophet()
        
        if verbose:
            print("\n3. Training Exponential Smoothing model...")
        self._fit_exponential_smoothing()
        
        # Machine Learning Models
        if verbose:
            print("\n4. Training XGBoost model...")
        self._fit_xgboost()
        
        # Deep Learning Models
        if verbose:
            print("\n5. Training LSTM model...")
        self._fit_lstm()
        
        if verbose:
            print("\n6. Training Bidirectional LSTM model...")
        self._fit_bilstm()
        
        # Ensemble
        if verbose:
            print("\n7. Creating Ensemble model...")
        self._create_ensemble()
        
        if verbose:
            print("\n✅ All models trained successfully!")
    
    def _fit_auto_arima(self):
        """Fit AutoARIMA model."""
        try:
            model = pm.auto_arima(
                self.train,
                seasonal=True,
                m=12,  # Monthly seasonality
                stepwise=True,
                suppress_warnings=True,
                error_action='ignore',
                trace=False,
                n_jobs=-1
            )
            
            # Generate forecasts
            forecast, conf_int = model.predict(
                n_periods=self.forecast_horizon,
                return_conf_int=True,
                alpha=1 - self.confidence_level
            )
            
            # Create forecast index
            last_date = self.train.index[-1]
            freq = pd.infer_freq(self.train.index) or 'D'
            forecast_index = pd.date_range(
                start=last_date + pd.Timedelta(1, freq[0] if freq else 'D'),
                periods=self.forecast_horizon,
                freq=freq
            )
            
            self.models['arima'] = model
            self.forecasts['arima'] = pd.Series(forecast, index=forecast_index)
            self.prediction_intervals['arima'] = conf_int
            
        except Exception as e:
            print(f"ARIMA model failed: {e}")
            self.models['arima'] = None
    
    def _fit_prophet(self):
        """Fit Prophet model."""
        try:
            # Prepare data for Prophet
            df = pd.DataFrame({
                'ds': self.train.index,
                'y': self.train.values
            })
            
            # Initialize and fit model
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=True,
                daily_seasonality=False,
                changepoint_prior_scale=0.05,
                seasonality_prior_scale=10,
                interval_width=self.confidence_level
            )
            
            # Add custom seasonalities if needed
            model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
            
            model.fit(df, verbose=False)
            
            # Make future dataframe
            future = model.make_future_dataframe(periods=self.forecast_horizon)
            forecast = model.predict(future)
            
            # Extract forecasts
            forecast_values = forecast['yhat'].iloc[-self.forecast_horizon:]
            forecast_index = pd.to_datetime(forecast['ds'].iloc[-self.forecast_horizon:])
            
            self.models['prophet'] = model
            self.forecasts['prophet'] = pd.Series(
                forecast_values.values,
                index=forecast_index
            )
            self.prediction_intervals['prophet'] = {
                'lower': forecast['yhat_lower'].iloc[-self.forecast_horizon:].values,
                'upper': forecast['yhat_upper'].iloc[-self.forecast_horizon:].values
            }
            
        except Exception as e:
            print(f"Prophet model failed: {e}")
            self.models['prophet'] = None
    
    def _fit_exponential_smoothing(self):
        """Fit Exponential Smoothing model."""
        try:
            # Determine best configuration
            model = ExponentialSmoothing(
                self.train,
                seasonal_periods=12,
                trend='add',
                seasonal='add',
                damped_trend=True
            )
            
            fitted = model.fit(optimized=True)
            
            # Generate forecasts
            forecast = fitted.forecast(self.forecast_horizon)
            
            # Generate prediction intervals using simulation
            simulations = fitted.simulate(
                nsimulations=self.forecast_horizon,
                repetitions=1000
            )
            
            lower = np.percentile(simulations, (1 - self.confidence_level) / 2 * 100, axis=1)
            upper = np.percentile(simulations, (1 + self.confidence_level) / 2 * 100, axis=1)
            
            self.models['exponential_smoothing'] = fitted
            self.forecasts['exponential_smoothing'] = forecast
            self.prediction_intervals['exponential_smoothing'] = {
                'lower': lower,
                'upper': upper
            }
            
        except Exception as e:
            print(f"Exponential Smoothing model failed: {e}")
            self.models['exponential_smoothing'] = None
    
    def _fit_xgboost(self, lags: int = 30):
        """Fit XGBoost model with feature engineering."""
        try:
            # Create lagged features
            def create_features(series, lags):
                df = pd.DataFrame(index=series.index)
                df['value'] = series.values
                
                # Lag features
                for lag in range(1, lags + 1):
                    df[f'lag_{lag}'] = df['value'].shift(lag)
                
                # Rolling statistics
                for window in [7, 14, 30]:
                    df[f'rolling_mean_{window}'] = df['value'].rolling(window).mean()
                    df[f'rolling_std_{window}'] = df['value'].rolling(window).std()
                
                # Time features
                df['month'] = df.index.month
                df['day_of_week'] = df.index.dayofweek
                df['day_of_month'] = df.index.day
                df['quarter'] = df.index.quarter
                
                return df.dropna()
            
            # Prepare training data
            train_df = create_features(self.train, lags)
            X_train = train_df.drop('value', axis=1)
            y_train = train_df['value']
            
            # Train model
            model = xgb.XGBRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=5,
                random_state=42
            )
            
            model.fit(X_train, y_train)
            
            # Generate forecasts iteratively
            last_values = self.train.iloc[-lags:].values.tolist()
            forecasts = []
            
            for _ in range(self.forecast_horizon):
                # Create features for prediction
                features = {}
                for i in range(1, lags + 1):
                    features[f'lag_{i}'] = last_values[-i]
                
                # Add rolling features
                for window in [7, 14, 30]:
                    if len(last_values) >= window:
                        features[f'rolling_mean_{window}'] = np.mean(last_values[-window:])
                        features[f'rolling_std_{window}'] = np.std(last_values[-window:])
                    else:
                        features[f'rolling_mean_{window}'] = np.mean(last_values)
                        features[f'rolling_std_{window}'] = np.std(last_values)
                
                # Add time features (simplified)
                features['month'] = 1
                features['day_of_week'] = 1
                features['day_of_month'] = 1
                features['quarter'] = 1
                
                # Make prediction
                X_pred = pd.DataFrame([features])[X_train.columns]
                pred = model.predict(X_pred)[0]
                
                forecasts.append(pred)
                last_values.append(pred)
            
            # Create forecast series
            last_date = self.train.index[-1]
            freq = pd.infer_freq(self.train.index) or 'D'
            forecast_index = pd.date_range(
                start=last_date + pd.Timedelta(1, freq[0] if freq else 'D'),
                periods=self.forecast_horizon,
                freq=freq
            )
            
            self.models['xgboost'] = model
            self.forecasts['xgboost'] = pd.Series(forecasts, index=forecast_index)
            
        except Exception as e:
            print(f"XGBoost model failed: {e}")
            self.models['xgboost'] = None
    
    def _fit_lstm(self, sequence_length: int = 30):
        """Fit LSTM model for time series."""
        try:
            # Prepare data
            scaler = MinMaxScaler()
            scaled_data = scaler.fit_transform(self.train.values.reshape(-1, 1))
            
            # Create sequences
            def create_sequences(data, seq_length):
                X, y = [], []
                for i in range(seq_length, len(data)):
                    X.append(data[i-seq_length:i, 0])
                    y.append(data[i, 0])
                return np.array(X), np.array(y)
            
            X_train, y_train = create_sequences(scaled_data, sequence_length)
            X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
            
            # Build model
            model = Sequential([
                LSTM(50, return_sequences=True, input_shape=(sequence_length, 1)),
                Dropout(0.2),
                LSTM(50, return_sequences=True),
                Dropout(0.2),
                LSTM(50),
                Dropout(0.2),
                Dense(1)
            ])
            
            model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
            
            # Callbacks
            early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='loss', patience=5, factor=0.5)
            
            # Train model
            model.fit(
                X_train, y_train,
                epochs=50,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop, reduce_lr]
            )
            
            # Generate forecasts
            last_sequence = scaled_data[-sequence_length:]
            forecasts = []
            
            for _ in range(self.forecast_horizon):
                next_pred = model.predict(last_sequence.reshape(1, sequence_length, 1), verbose=0)
                forecasts.append(next_pred[0, 0])
                last_sequence = np.append(last_sequence[1:], next_pred)
            
            # Inverse transform
            forecasts = scaler.inverse_transform(np.array(forecasts).reshape(-1, 1)).flatten()
            
            # Create forecast series
            last_date = self.train.index[-1]
            freq = pd.infer_freq(self.train.index) or 'D'
            forecast_index = pd.date_range(
                start=last_date + pd.Timedelta(1, freq[0] if freq else 'D'),
                periods=self.forecast_horizon,
                freq=freq
            )
            
            self.models['lstm'] = {'model': model, 'scaler': scaler}
            self.forecasts['lstm'] = pd.Series(forecasts, index=forecast_index)
            
        except Exception as e:
            print(f"LSTM model failed: {e}")
            self.models['lstm'] = None
    
    def _fit_bilstm(self, sequence_length: int = 30):
        """Fit Bidirectional LSTM model."""
        try:
            # Prepare data
            scaler = MinMaxScaler()
            scaled_data = scaler.fit_transform(self.train.values.reshape(-1, 1))
            
            # Create sequences
            def create_sequences(data, seq_length):
                X, y = [], []
                for i in range(seq_length, len(data)):
                    X.append(data[i-seq_length:i, 0])
                    y.append(data[i, 0])
                return np.array(X), np.array(y)
            
            X_train, y_train = create_sequences(scaled_data, sequence_length)
            X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
            
            # Build model with attention
            inputs = Input(shape=(sequence_length, 1))
            
            # Bidirectional LSTM layers
            x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
            x = Dropout(0.2)(x)
            x = Bidirectional(LSTM(32, return_sequences=True))(x)
            x = Dropout(0.2)(x)
            
            # Attention mechanism
            attention = tf.keras.layers.MultiHeadAttention(
                num_heads=4,
                key_dim=32
            )(x, x)
            
            # Final layers
            x = tf.keras.layers.GlobalAveragePooling1D()(attention)
            x = Dense(32, activation='relu')(x)
            x = Dropout(0.2)(x)
            outputs = Dense(1)(x)
            
            model = Model(inputs=inputs, outputs=outputs)
            model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
            
            # Train model
            model.fit(
                X_train, y_train,
                epochs=50,
                batch_size=32,
                verbose=0,
                callbacks=[
                    EarlyStopping(monitor='loss', patience=10, restore_best_weights=True),
                    ReduceLROnPlateau(monitor='loss', patience=5, factor=0.5)
                ]
            )
            
            # Generate forecasts
            last_sequence = scaled_data[-sequence_length:]
            forecasts = []
            
            for _ in range(self.forecast_horizon):
                next_pred = model.predict(last_sequence.reshape(1, sequence_length, 1), verbose=0)
                forecasts.append(next_pred[0, 0])
                last_sequence = np.append(last_sequence[1:], next_pred)
            
            # Inverse transform
            forecasts = scaler.inverse_transform(np.array(forecasts).reshape(-1, 1)).flatten()
            
            # Create forecast series
            last_date = self.train.index[-1]
            freq = pd.infer_freq(self.train.index) or 'D'
            forecast_index = pd.date_range(
                start=last_date + pd.Timedelta(1, freq[0] if freq else 'D'),
                periods=self.forecast_horizon,
                freq=freq
            )
            
            self.models['bilstm'] = {'model': model, 'scaler': scaler}
            self.forecasts['bilstm'] = pd.Series(forecasts, index=forecast_index)
            
        except Exception as e:
            print(f"Bidirectional LSTM model failed: {e}")
            self.models['bilstm'] = None
    
    def _create_ensemble(self, weights: Dict[str, float] = None):
        """Create ensemble forecast from individual models."""
        try:
            # Get available forecasts
            available_models = [k for k, v in self.forecasts.items() if v is not None]
            
            if not available_models:
                print("No models available for ensemble")
                return
            
            # Default equal weights
            if weights is None:
                weights = {model: 1.0 / len(available_models) for model in available_models}
            
            # Weighted average
            ensemble_forecast = None
            for model in available_models:
                if model in weights:
                    weight = weights[model]
                    if ensemble_forecast is None:
                        ensemble_forecast = self.forecasts[model] * weight
                    else:
                        ensemble_forecast += self.forecasts[model] * weight
            
            self.forecasts['ensemble'] = ensemble_forecast
            self.models['ensemble'] = {'models': available_models, 'weights': weights}
            
        except Exception as e:
            print(f"Ensemble creation failed: {e}")
            self.models['ensemble'] = None

## 2. Model Evaluation Class

In [ ]:
class ModelEvaluator:
    """Evaluate and compare forecasting models."""
    
    def __init__(self, actual: pd.Series, forecasts: Dict[str, pd.Series]):
        """Initialize the evaluator.
        
        Parameters:
        -----------
        actual : pd.Series
            Actual values
        forecasts : Dict[str, pd.Series]
            Dictionary of model forecasts
        """
        self.actual = actual
        self.forecasts = forecasts
        
    def evaluate_all_models(self) -> pd.DataFrame:
        """Evaluate all forecasting models."""
        
        results = []
        
        for model_name, forecast in self.forecasts.items():
            if forecast is None:
                continue
                
            # Align forecast with actual values
            common_index = self.actual.index.intersection(forecast.index)
            if len(common_index) == 0:
                continue
                
            actual_aligned = self.actual.loc[common_index]
            forecast_aligned = forecast.loc[common_index]
            
            # Calculate metrics
            mae = mean_absolute_error(actual_aligned, forecast_aligned)
            rmse = np.sqrt(mean_squared_error(actual_aligned, forecast_aligned))
            mape = mean_absolute_percentage_error(actual_aligned, forecast_aligned) * 100
            
            # Directional accuracy
            if len(actual_aligned) > 1:
                direction_actual = np.diff(actual_aligned) > 0
                direction_pred = np.diff(forecast_aligned) > 0
                directional_accuracy = np.mean(direction_actual == direction_pred) * 100
            else:
                directional_accuracy = 0
            
            # Theil's U statistic
            naive_forecast = actual_aligned.shift(1).fillna(actual_aligned.mean())
            mse_model = mean_squared_error(actual_aligned, forecast_aligned)
            mse_naive = mean_squared_error(actual_aligned, naive_forecast)
            theils_u = np.sqrt(mse_model / mse_naive) if mse_naive > 0 else np.inf
            
            results.append({
                'Model': model_name,
                'MAE': mae,
                'RMSE': rmse,
                'MAPE (%)': mape,
                'Directional Accuracy (%)': directional_accuracy,
                "Theil's U": theils_u
            })
        
        return pd.DataFrame(results).sort_values('RMSE')
    
    def plot_forecast_comparison(self) -> go.Figure:
        """Plot all forecasts against actual values."""
        
        fig = go.Figure()
        
        # Plot actual values
        fig.add_trace(go.Scatter(
            x=self.actual.index,
            y=self.actual.values,
            mode='lines',
            name='Actual',
            line=dict(color='black', width=2)
        ))
        
        # Color palette
        colors = px.colors.qualitative.Plotly
        
        # Plot forecasts
        for i, (model_name, forecast) in enumerate(self.forecasts.items()):
            if forecast is not None:
                fig.add_trace(go.Scatter(
                    x=forecast.index,
                    y=forecast.values,
                    mode='lines',
                    name=model_name,
                    line=dict(color=colors[i % len(colors)])
                ))
        
        fig.update_layout(
            title="Forecast Comparison",
            xaxis_title="Date",
            yaxis_title="Value",
            hovermode='x unified',
            height=500
        )
        
        return fig
    
    def plot_residuals(self) -> go.Figure:
        """Plot residuals for each model."""
        
        n_models = len([f for f in self.forecasts.values() if f is not None])
        fig = make_subplots(
            rows=n_models, cols=1,
            subplot_titles=list(self.forecasts.keys()),
            vertical_spacing=0.05
        )
        
        row = 1
        for model_name, forecast in self.forecasts.items():
            if forecast is None:
                continue
                
            # Align forecast with actual values
            common_index = self.actual.index.intersection(forecast.index)
            if len(common_index) == 0:
                continue
                
            actual_aligned = self.actual.loc[common_index]
            forecast_aligned = forecast.loc[common_index]
            residuals = actual_aligned - forecast_aligned
            
            # Plot residuals
            fig.add_trace(
                go.Scatter(
                    x=common_index,
                    y=residuals,
                    mode='lines',
                    name=model_name
                ),
                row=row, col=1
            )
            
            # Add zero line
            fig.add_hline(y=0, line_dash="dash", line_color="red", row=row, col=1)
            
            row += 1
        
        fig.update_layout(height=200 * n_models, title="Residual Analysis", showlegend=False)
        fig.update_xaxes(title_text="Date", row=n_models, col=1)
        
        return fig

## 3. Generate Sample Data and Train Models

In [ ]:
# Generate sample time series data
np.random.seed(42)

# Create date range
dates = pd.date_range(start='2020-01-01', end='2023-12-31', freq='D')

# Generate time series with multiple components
trend = np.linspace(100, 150, len(dates))
seasonal_annual = 10 * np.sin(2 * np.pi * np.arange(len(dates)) / 365.25)
seasonal_monthly = 5 * np.sin(2 * np.pi * np.arange(len(dates)) / 30.44)
weekly = 3 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)
noise = np.random.normal(0, 5, len(dates))

# Combine components
values = trend + seasonal_annual + seasonal_monthly + weekly + noise

# Add some non-linear patterns
values = values + 0.001 * (np.arange(len(dates)) ** 1.2)

# Create series
ts = pd.Series(values, index=dates, name='value')

# Split into train and test
split_date = '2023-10-01'
train_ts = ts[ts.index < split_date]
test_ts = ts[ts.index >= split_date]

print(f"Time series created:")
print(f"  - Total observations: {len(ts)}")
print(f"  - Training set: {len(train_ts)} observations ({train_ts.index.min()} to {train_ts.index.max()})")
print(f"  - Test set: {len(test_ts)} observations ({test_ts.index.min()} to {test_ts.index.max()})")

# Plot the data
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_ts.index, y=train_ts.values, mode='lines', name='Training Data'))
fig.add_trace(go.Scatter(x=test_ts.index, y=test_ts.values, mode='lines', name='Test Data'))
fig.update_layout(title="Time Series Data", xaxis_title="Date", yaxis_title="Value", height=400)
fig.show()

## 4. Train All Models

In [ ]:
# Initialize forecaster
forecaster = AdvancedForecaster(
    train_data=train_ts,
    test_data=test_ts,
    forecast_horizon=len(test_ts),
    confidence_level=0.95
)

# Train all models
forecaster.fit_all_models(verbose=True)

## 5. Evaluate Models

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluator(
    actual=test_ts,
    forecasts=forecaster.forecasts
)

# Get evaluation metrics
metrics_df = evaluator.evaluate_all_models()

print("\n" + "="*80)
print("MODEL PERFORMANCE METRICS")
print("="*80)
print(metrics_df.to_string())

# Highlight best model
best_model = metrics_df.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_model} (lowest RMSE)")

## 6. Visualize Results

In [ ]:
# Plot forecast comparison
fig = evaluator.plot_forecast_comparison()
fig.show()

In [ ]:
# Plot residuals
fig = evaluator.plot_residuals()
fig.show()

## 7. Forecast with Confidence Intervals

In [ ]:
# Plot forecasts with confidence intervals for models that support them
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['ARIMA with Confidence Intervals', 'Prophet with Confidence Intervals'],
    vertical_spacing=0.1,
    shared_xaxes=True
)

# ARIMA
if 'arima' in forecaster.forecasts and forecaster.forecasts['arima'] is not None:
    forecast = forecaster.forecasts['arima']
    
    # Plot training data
    fig.add_trace(
        go.Scatter(x=train_ts.index, y=train_ts.values, mode='lines', 
                  name='Training Data', line=dict(color='blue')),
        row=1, col=1
    )
    
    # Plot forecast
    fig.add_trace(
        go.Scatter(x=forecast.index, y=forecast.values, mode='lines',
                  name='ARIMA Forecast', line=dict(color='red')),
        row=1, col=1
    )
    
    # Plot confidence intervals if available
    if 'arima' in forecaster.prediction_intervals and forecaster.prediction_intervals['arima'] is not None:
        conf_int = forecaster.prediction_intervals['arima']
        fig.add_trace(
            go.Scatter(
                x=forecast.index.tolist() + forecast.index.tolist()[::-1],
                y=conf_int[:, 0].tolist() + conf_int[:, 1].tolist()[::-1],
                fill='toself',
                fillcolor='rgba(255,0,0,0.2)',
                line=dict(color='rgba(255,0,0,0)'),
                name='95% CI'
            ),
            row=1, col=1
        )

# Prophet
if 'prophet' in forecaster.forecasts and forecaster.forecasts['prophet'] is not None:
    forecast = forecaster.forecasts['prophet']
    
    # Plot training data
    fig.add_trace(
        go.Scatter(x=train_ts.index, y=train_ts.values, mode='lines',
                  name='Training Data', line=dict(color='blue')),
        row=2, col=1
    )
    
    # Plot forecast
    fig.add_trace(
        go.Scatter(x=forecast.index, y=forecast.values, mode='lines',
                  name='Prophet Forecast', line=dict(color='green')),
        row=2, col=1
    )
    
    # Plot confidence intervals if available
    if 'prophet' in forecaster.prediction_intervals and forecaster.prediction_intervals['prophet'] is not None:
        conf_int = forecaster.prediction_intervals['prophet']
        fig.add_trace(
            go.Scatter(
                x=forecast.index.tolist() + forecast.index.tolist()[::-1],
                y=conf_int['lower'].tolist() + conf_int['upper'].tolist()[::-1],
                fill='toself',
                fillcolor='rgba(0,255,0,0.2)',
                line=dict(color='rgba(0,255,0,0)'),
                name='95% CI'
            ),
            row=2, col=1
        )

fig.update_layout(height=700, title="Forecasts with Confidence Intervals", showlegend=True)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_yaxes(title_text="Value", row=2, col=1)
fig.show()

## 8. Feature Importance (XGBoost)

In [ ]:
# Display feature importance for XGBoost model
if 'xgboost' in forecaster.models and forecaster.models['xgboost'] is not None:
    model = forecaster.models['xgboost']
    
    # Get feature importance
    importance = model.feature_importances_
    feature_names = [f'Feature_{i}' for i in range(len(importance))]
    
    # Create dataframe
    importance_df = pd.DataFrame({
        'Feature': feature_names[:20],  # Top 20 features
        'Importance': importance[:20]
    }).sort_values('Importance', ascending=True)
    
    # Plot feature importance
    fig = go.Figure(go.Bar(
        x=importance_df['Importance'],
        y=importance_df['Feature'],
        orientation='h'
    ))
    
    fig.update_layout(
        title="XGBoost Feature Importance (Top 20)",
        xaxis_title="Importance",
        yaxis_title="Feature",
        height=500
    )
    
    fig.show()

## 9. Save Models and Results

In [ ]:
# Save results to CSV
metrics_df.to_csv('forecast_metrics.csv', index=False)
print("✅ Metrics saved to 'forecast_metrics.csv'")

# Save forecasts
forecasts_df = pd.DataFrame(forecaster.forecasts)
forecasts_df.to_csv('forecasts.csv')
print("✅ Forecasts saved to 'forecasts.csv'")

# Save best model (example with ARIMA)
import pickle

if best_model == 'arima' and 'arima' in forecaster.models:
    with open('best_model_arima.pkl', 'wb') as f:
        pickle.dump(forecaster.models['arima'], f)
    print(f"✅ Best model ({best_model}) saved to 'best_model_arima.pkl'")

print("\n📊 Analysis complete!")